# 05 – DQ Export Tables (schema `Exports`)
For every active rule, writes the rule's **full population** (`base_sql`, all columns) to one Delta table, with a
flag column:

- `dq_status` = `INVALID` when the row meets the rule's `failure_condition`, otherwise `VALID`
  (rules that use `failure_sql`: `INVALID` when the row's `record_key` is in the failure query's result)
- `dq_rule_id`, `dq_exported_at`

Tables: `Customer_MDM_PoC.Exports.dq_rule_<id>` (e.g. `dq_rule_008`), **overwritten on every run** (latest state).
Named by rule ID only, so renaming a rule does not create a new table and break Power BI.
A summary row per rule goes to `Exports.export_table_log`.

Run it in the pipeline after `03_dq_runner` (it can run in parallel with `04_dq_export`). It changes no DQ table.

Note: these tables hold **rows**; dashboard counts are distinct records (`record_key`), so they can differ.

In [ ]:
%run 00_dq_config

In [ ]:
# Step 1: settings (fallbacks, so this works even if 00_dq_config was not re-imported)
import re

EXPORT_SCHEMA      = globals().get("EXPORT_TABLES_FQN", f"{LAKEHOUSE}.Exports")
T_EXPORT_TABLE_LOG = f"{EXPORT_SCHEMA}.export_table_log"
STATUS_COL         = "dq_status"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {EXPORT_SCHEMA}")
export_ts = now_local()

def clean_columns(df):
    # Delta column names: letters, digits and _ only; duplicates (e.g. from SELECT * over joins) get _2, _3 ...
    names, used = [], set()
    for c in df.columns:
        n = re.sub(r"[^A-Za-z0-9_]+", "_", c).strip("_") or "col"
        cand, i = n, 1
        while cand.lower() in used:
            i += 1
            cand = f"{n}_{i}"
        used.add(cand.lower())
        names.append(cand)
    return df.toDF(*names)

In [ ]:
# Step 2: one table per active rule = full population + VALID / INVALID flag
rules = (spark.table(T_CATALOG)
              .filter("is_active = true")
              .orderBy("rule_id")
              .collect())

log_rows = []

for r in rules:
    table = f"{EXPORT_SCHEMA}.dq_rule_{r.rule_id:03d}"
    t0 = time.time()
    try:
        base = spark.sql(r.base_sql)

        if r.failure_condition:
            # NULL result of the condition counts as VALID - same as the runner, where filter() drops NULLs
            flagged = base.withColumn(STATUS_COL,
                                      F.when(F.expr(r.failure_condition), F.lit("INVALID")).otherwise(F.lit("VALID")))
        else:
            bad_keys = (spark.sql(r.failure_sql)
                             .select(F.col("record_key").cast("string").alias("_dq_bad_key"))
                             .distinct()
                             .withColumn("_dq_is_bad", F.lit(True)))
            flagged = (base.withColumn("_dq_key", F.col("record_key").cast("string"))
                           .join(bad_keys, F.col("_dq_key") == F.col("_dq_bad_key"), "left")
                           .withColumn(STATUS_COL,
                                       F.when(F.col("_dq_is_bad"), F.lit("INVALID")).otherwise(F.lit("VALID")))
                           .drop("_dq_key", "_dq_bad_key", "_dq_is_bad"))

        flagged = (clean_columns(flagged)
                       .withColumn("dq_rule_id", F.lit(r.rule_id).cast("int"))
                       .withColumn("dq_exported_at", F.lit(export_ts).cast("timestamp")))

        (flagged.write.mode("overwrite").option("overwriteSchema", "true")
                .format("delta").saveAsTable(table))

        counts  = {x[STATUS_COL]: x["count"] for x in spark.table(table).groupBy(STATUS_COL).count().collect()}
        invalid = int(counts.get("INVALID", 0))
        valid   = int(counts.get("VALID", 0))
        log_rows.append((r.rule_id, r.rule_name, table, valid + invalid, invalid, valid,
                         export_ts, "SUCCESS", None, round(time.time() - t0, 1)))
        print(f"Rule {r.rule_id}: {table}  rows={valid + invalid}  invalid={invalid}")

    except Exception as e:
        log_rows.append((r.rule_id, r.rule_name, table, None, None, None,
                         export_ts, "FAILED", str(e)[:2000], round(time.time() - t0, 1)))
        print(f"Rule {r.rule_id} FAILED: {str(e)[:300]}")

In [ ]:
# Step 3: summary per rule (latest run only) -> Exports.export_table_log
log_schema = ("rule_id INT, rule_name STRING, table_name STRING, total_rows BIGINT, invalid_rows BIGINT, "
              "valid_rows BIGINT, exported_at TIMESTAMP, status STRING, error_message STRING, duration_sec DOUBLE")
if log_rows:
    (spark.createDataFrame(log_rows, log_schema)
          .write.mode("overwrite").option("overwriteSchema", "true")
          .format("delta").saveAsTable(T_EXPORT_TABLE_LOG))
    display(spark.table(T_EXPORT_TABLE_LOG).orderBy("rule_id"))

In [ ]:
# Step 4: return result to the pipeline
n_ok  = sum(1 for x in log_rows if x[7] == "SUCCESS")
n_err = len(log_rows) - n_ok
notebookutils.notebook.exit(f"tables={n_ok}; errors={n_err}")